In [854]:
board = [
[0,0,0],
[0,0,0],
[0,0,0]
]



In [855]:
board

[[0, 0, 0], [0, 0, 0], [0, 0, 0]]

In [856]:
board[1][1] = 1

In [857]:
board

[[0, 0, 0], [0, 1, 0], [0, 0, 0]]

In [858]:
empty = 0
X = 1 
O = -1

In [859]:
board[0][0] =X

In [860]:
board[1][1] = O

In [861]:
board

[[1, 0, 0], [0, -1, 0], [0, 0, 0]]

In [862]:
def invalid(a,b):
    if board[a][b] == X or board[a][b] == O:
        return True

    return False

In [863]:
def move(current_player,a,b):
  if current_player == X:
    board[a][b] = 1
  else:
     board[a][b] = -1


In [864]:
def clear(board):
  for i in range (3):
    for j in range (3):
      board[i][j] = 0

In [865]:
clear(board)

In [866]:
board  

[[0, 0, 0], [0, 0, 0], [0, 0, 0]]

In [867]:
current_player = X
move(X,0,0)

In [868]:
board

[[1, 0, 0], [0, 0, 0], [0, 0, 0]]

In [869]:
board

[[1, 0, 0], [0, 0, 0], [0, 0, 0]]

In [870]:
def check_win(board):
  if board[0][0] == board[0][1] == board [0][2] == 1 or board[1][0] == board[1][1] == board [1][2] == 1 or board[2][0] == board[2][1] == board [2][2] == 1:
    return X
  elif board[0][0] == board[1][0] == board [2][0] == 1 or board[0][1] == board[1][1] == board [2][1] == 1 or board[0][2] == board[1][2] == board [2][2] == 1:
    return X
  elif board[0][0] == board[1][1] == board[2][2] == 1:
    return X
  elif board[0][2] == board[1][1] == board[2][0] == 1:
    return X
  elif board[0][0] == board[0][1] == board [0][2] == -1 or board[1][0] == board[1][1] == board [1][2] == -1 or board[2][0] == board[2][1] == board [2][2] == -1:
    return O
  elif board[0][0] == board[1][0] == board [2][0] == -1 or board[0][1] == board[1][1] == board [2][1] == -1 or board[0][2] == board[1][2] == board [2][2] == -1:
    return O
  elif board[0][0] == board[1][1] == board[2][2] == -1:
    return O
  elif board[0][2] == board[1][1] == board[2][0] == -1:
    return O
  else:
    return None

In [871]:
Draw = "Draw"
def check_draw(board):
  board_full = True
  for i in range (3):
    for j in range (3):
      if board[i][j] == 0:
        board_full= False
 
  if check_win(board) == None and board_full:
    return Draw
  

In [872]:
import random
def random_move():

  row = random.randint(0,2)
  column = random.randint(0,2)
  return row,column

In [873]:
clear(board)
print(board)


[[0, 0, 0], [0, 0, 0], [0, 0, 0]]


In [874]:
def flatten(board):
  new_board = []
  for i in range (3):
    for j in range (3):
      new_board.append(board[i][j])
  return new_board 


In [875]:
def indexing(row,column):
  index = row*3 + column
  return index


In [876]:
X_train = []
y_train = []


In [877]:
for game in range (5000):
  clear(board)
  history = []
  current_player = X
  while (check_win(board)!= X and check_win(board)!= O) and check_draw(board)!=Draw:
    board_snapshot = [row[:] for row in board]
    row , column = random_move()

    while invalid(row,column):
      row , column = random_move()
    if current_player == X:
      history.append([board_snapshot,
                (row,column),
                X])
      move(X, row, column)
      
      current_player = O

    else:
      history.append([board_snapshot,
                (row,column),
                O])
      move(O,row,column)
      
      current_player = X

  if check_win(board) == X:
    reward = 1

  elif check_win(board) == O:
    reward = -1

  elif check_draw(board) == Draw:
    reward = 0
  for i in history:
    i.append(reward)
  winner_moves = []
  for i in range (len(history)):
    if (reward==1 and history[i][2] == 1):
      winner_moves.append(history[i])
    elif (reward==-1 and history[i][2] == -1):
       winner_moves.append(history[i])

  if len(winner_moves) > 0:
    last_move = winner_moves[-3:]

    for moves in last_move:
      X_train.append(flatten(moves[0]))
      y_train.append(indexing(*moves[1]))  





In [878]:
#X_train

In [879]:
#y_train

In [880]:
print(len(X_train))
print(len(y_train))

13170
13170


In [881]:
import torch

In [882]:
X_train = torch.tensor(X_train)
y_train = torch.tensor(y_train)

In [883]:
print(X_train.shape)
print(y_train.shape)

torch.Size([13170, 9])
torch.Size([13170])


# Defining the model

In [884]:
# define NN class
import torch.nn as nn
class MyNN(nn.Module):
  def __init__(self, num_features):

    super().__init__()
    self.model = nn.Sequential(
      nn.Linear(num_features, 32),
      nn.ReLU(),
      nn.Linear(32,32),
      nn.ReLU(),
      nn.Linear(32, 9)
    )

  def forward(self, x):
      return self.model(x)

  

In [885]:
model = MyNN(9)

In [886]:
criterion = nn.CrossEntropyLoss()

In [887]:
import torch.optim as optim
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [888]:
X_train = X_train.float()
epochs = 100

# training loop

In [889]:
for epoch in range(epochs):
  y_pred = model(X_train)
  loss = criterion(y_pred, y_train)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')


Epoch: 1, Loss: 2.194035291671753
Epoch: 2, Loss: 2.1927452087402344
Epoch: 3, Loss: 2.191474199295044
Epoch: 4, Loss: 2.190228223800659
Epoch: 5, Loss: 2.1890060901641846
Epoch: 6, Loss: 2.187809467315674
Epoch: 7, Loss: 2.1866378784179688
Epoch: 8, Loss: 2.1854851245880127
Epoch: 9, Loss: 2.184357166290283
Epoch: 10, Loss: 2.1832516193389893
Epoch: 11, Loss: 2.182164430618286
Epoch: 12, Loss: 2.181100845336914
Epoch: 13, Loss: 2.1800577640533447
Epoch: 14, Loss: 2.17903470993042
Epoch: 15, Loss: 2.1780245304107666
Epoch: 16, Loss: 2.177029609680176
Epoch: 17, Loss: 2.1760504245758057
Epoch: 18, Loss: 2.1750869750976562
Epoch: 19, Loss: 2.174138307571411
Epoch: 20, Loss: 2.173204183578491
Epoch: 21, Loss: 2.172283172607422
Epoch: 22, Loss: 2.1713762283325195
Epoch: 23, Loss: 2.1704814434051514
Epoch: 24, Loss: 2.1695985794067383
Epoch: 25, Loss: 2.168728828430176
Epoch: 26, Loss: 2.167867660522461
Epoch: 27, Loss: 2.1670122146606445
Epoch: 28, Loss: 2.166161060333252
Epoch: 29, Loss: 

In [890]:
sample = X_train[0]

In [891]:
sample = sample.unsqueeze(0)

In [892]:
output = model(sample)
output

tensor([[ 0.0252, -0.2983,  0.1600, -0.0460,  0.4702, -0.3900,  0.0522, -0.4369,
          0.1303]], grad_fn=<AddmmBackward0>)

In [893]:
prediction = torch.argmax(output)
print(prediction)

tensor(4)


In [894]:
clear(board)
print(board)

[[0, 0, 0], [0, 0, 0], [0, 0, 0]]


In [895]:

ai_wins = 0
random_wins = 0
draws = 0

for game in range(100):
  clear(board)
  current_player = X
  while (check_win(board)!=X and check_win(board)!=O) and check_draw(board)!=Draw:
    if current_player == X:
      input = torch.tensor(flatten(board))
      input = input.float()
      input = input.unsqueeze(0)
      output = model(input)

      ranked_moves = torch.argsort(output, descending=True)

      for index in ranked_moves[0]:
        index = index.item()
        row = index // 3
        column = index % 3
        if not invalid(row,column):
         move(X,row,column)
        current_player = O
        break
    
    else:

      row,column = random_move()

      while invalid(row,column):
        row,column = random_move()
      
      move(O,row,column)

      current_player = X
    
    
  if check_win(board) == X:
    ai_wins += 1

  elif check_win(board) == O:
    random_wins += 1

  else:
    draws += 1

In [896]:

print(ai_wins)
print(random_wins)
print(draws)

86
14
0
